In [ ]:
%matplotlib inline
%matplotlib widget
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
"""
NASA Exoplanet Archive analysis
===========================================================

The archive table is NOT a homogeneous catalogue: for every planet it can contain
several *rows*, one per publication/analysis pipeline (soltype), plus a
`default_flag` marking which row the Archive currently recommends. It also
mixes together three fundamentally different kinds of "planet":

    1. Published Confirmed planets          -> vetted, peer-reviewed
    2. Published Candidate planets          -> not yet confirmed
    3. Mission-pipeline candidates          -> Kepler KOI (6 separate pipeline
       versions!) and TESS TOI candidates, i.e. automated vetting products
       that have NOT been confirmed and were run on different data releases.

Outputs (in ./exoplanet_analysis_outputs/):
    - homogeneity_breakdown.png
    - radius_distribution.png
    - mass_radius_scatter.png
    - system_architecture.png
    - super_earth_cold_giant_systems.png
    - summary_stats.txt
    - super_earth_cold_giant_systems.csv
"""
# ----------------------------------------------------------------------------
# 0. Setup and planet definitions
# ----------------------------------------------------------------------------

CSV_PATH ="/Users/claudia/Documents/PostDoc/NASA_exoplanet_archive/PS_2026.08.10_09.39.26.csv"
OUT_DIR = "figures/exoplanet_analysis_outputs"

# Planet classification definitions (Earth radii / Earth masses / AU).
SUPER_EARTH_RADIUS = (1.25, 2.0)     # Rp in Earth radii
SUPER_EARTH_MASS = (2.0, 10.0)       # fallback: Mp in Earth masses, if radius missing
COLD_GIANT_RADIUS_MIN = 6.0          # Rp >= this counts as "giant" (~ Neptune and up)
COLD_GIANT_MASS_MIN = 95.0           # fallback giant mass cut (~ 0.3 Mjup) in Earth masses
COLD_GIANT_SMA_MIN = 1.0             # a >= this (AU) counts as "cold" (beyond ~ snow line)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.labelsize": 15,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    # major ticks
    "xtick.major.size": 12,
    "ytick.major.size": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # minor ticks
    "xtick.minor.size": 8,
    "ytick.minor.size": 8,
    "xtick.minor.visible": True,   # minor ticks are hidden by default — need this to actually see them
    "ytick.minor.visible": True,

})

COLORS = {
    "confirmed": "#2a6f97",
    "candidate": "#e07a5f",
    "kepler": "#81b29a",
    "tess": "#f2cc8f",
    "super_earth": "#803d65",
    "cold_giant": "#4dd3ee",
    "both": "#6a4c93",
    "other": "#c9c9c9",
}

#ensure that the output directory exists
def ensure_out_dir():
    os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# ----------------------------------------------------------------------------
# Load the Nasa Exoplanet Archive CSV
# ----------------------------------------------------------------------------

def load_data(path):
    """Load the raw archive CSV, and skips the '#' metadata header block."""
    df = pd.read_csv(path, comment="#", low_memory=False)
    return df

# ----------------------------------------------------------------------------
# Split into homogeneous sub-catalogues
# ----------------------------------------------------------------------------

def split_by_homogeneity(df):
    """Return a dict of homogeneous sub-catalogues plus a printable summary.
    The flags used to split the archive table:
      - `default`       : one row per planet (Archive's chosen best solution),
                          still a MIX of confirmed + candidate planets.
      - `confirmed`     : default row AND soltype == Published Confirmed.
                          This is the only sub-catalogue solid enough for
                          robust population statistics.
      - `candidates_*`  : kept separate because they are unvalidated and each
                          pipeline version has different completeness/purity.
    """
    catalogues = {}

    catalogues["raw_all_rows"] = df
    # the default solution for each planet (default_flag == 1) is the Archive's recommended row
    default_rows = df[df["default_flag"] == 1].copy()
    catalogues["default"] = default_rows
    # the published planets that ARE confirmed (out of the default rows, which are a mix of confirmed + candidates)
    is_confirmed = default_rows["soltype"].eq("Published Confirmed")
    catalogues["confirmed"] = default_rows[is_confirmed].copy()
    # the published planets that are NOT confirmed (i.e. still candidates, out of the default rows, which are a mix of confirmed + candidates)
    is_published_candidate = default_rows["soltype"].eq("Published Candidate")
    catalogues["published_candidates"] = default_rows[is_published_candidate].copy()
    # Kepler Objects of Interest (KOI) pipeline candidates, which are unvalidated
    is_kepler = default_rows["soltype"].str.startswith("Kepler", na=False)
    catalogues["kepler_pipeline_candidates"] = default_rows[is_kepler].copy()
    # TESS Objects of Interest (TOI) pipeline candidates, which are unvalidated
    is_tess = default_rows["soltype"].eq("TESS Project Candidate")
    catalogues["tess_pipeline_candidates"] = default_rows[is_tess].copy()

    return catalogues


def summarize_homogeneity(df, catalogues):
    """All the numbers in here are for formatting"""
    lines = []
    lines.append("=" * 70)
    lines.append("CATALOGUE HOMOGENEITY BREAKDOWN")
    lines.append("=" * 70)
    lines.append(f"Rows in NASA archive (all solutions, all pipelines): {len(df):,}")
    lines.append(f"Total number of unique planets: {df['pl_name'].nunique():,}")
    lines.append("")
    lines.append("Rows by soltype (before picking default solution):")
    for k, v in df["soltype"].value_counts().items():
        lines.append(f"  {k:<48} {v:>6,}") #this is just formatting the output nicely
    lines.append("")
    lines.append("Rows by soltype after picking default solution (default_flag == 1):")
    lines.append(f"  total default rows: {len(catalogues['default']):>35,}")
    for k, v in catalogues["default"]["soltype"].value_counts().items():
        lines.append(f"  {k:<48} {v:>6,}")
    lines.append("")
    lines.append(f"-> 'confirmed' science catalogue: "
                  f"{len(catalogues['confirmed']):,} planets")
    lines.append("   (Published Confirmed only, one row per planet)")
    text = "\n".join(lines)
    print(text)
    return text

# visualise the catalogue content
df = load_data(CSV_PATH)
catalogues = split_by_homogeneity(df)
summarize_homogeneity(df=df, catalogues=catalogues)

In [ ]:
# ----------------------------------------------------------------------------
# Classify planets: Super-Earths and Cold Giants
# ----------------------------------------------------------------------------

# creates a copy of the df and adds a new column (pl_category) to classify planets 
# as Super-Earths or Cold Giants based on their radius, mass, and semi-major axis.

def classify_planets(df):
    df = df.copy()
    # radius, mass and semi-major axis
    rade = df["pl_rade"]
    bmasse = df["pl_bmasse"]
    sma = df["pl_orbsmax"]
    # first SE classification is based on radius, then fallback to mass-based classification only where radius is missing
    is_super_earth = rade.between(*SUPER_EARTH_RADIUS)
    # fallback to mass-based classification only where radius is missing
    mass_fallback = rade.isna() & bmasse.between(*SUPER_EARTH_MASS)
    is_super_earth = is_super_earth | mass_fallback

    is_giant_size = (rade >= COLD_GIANT_RADIUS_MIN) | (
        rade.isna() & (bmasse >= COLD_GIANT_MASS_MIN)
    )
    is_cold = sma >= COLD_GIANT_SMA_MIN
    is_cold_giant = is_giant_size & is_cold

    df["is_super_earth"] = is_super_earth.fillna(False) # subsitutes the Nan values (planet with both radius and mass missing) with False, so they are not SE
    df["is_cold_giant"] = is_cold_giant.fillna(False) # subsitutes the Nan values (planet with both radius and mass missing) with False, so they are not CG

    def label(row):
        if row["is_super_earth"] and row["is_cold_giant"]:
            return "both (ambiguous)"
        if row["is_super_earth"]:
            return "super_earth"
        if row["is_cold_giant"]:
            return "cold_giant"
        return "other"

    df["pl_category"] = df.apply(label, axis=1)
    return df

def print_planets_stats(df):
    lines = []
    lines.append("=" * 70)
    lines.append("PLANET-LEVEL STATISTICS  (confirmed, default-solution catalogue)")
    lines.append("=" * 70)
    n_total = len(df)
    n_se = df["is_super_earth"].sum()
    n_cg = df["is_cold_giant"].sum()
    lines.append(f"Total confirmed planets analysed: {n_total:,}")
    lines.append(f"Super-Earths ({SUPER_EARTH_RADIUS[0]}-{SUPER_EARTH_RADIUS[1]} Re): "
                 f"{n_se:,}  ({100*n_se/n_total:.1f}%)")
    lines.append(f"Cold giants (R>={COLD_GIANT_RADIUS_MIN} Re & a>={COLD_GIANT_SMA_MIN} AU): "
                 f"{n_cg:,}  ({100*n_cg/n_total:.1f}%)")
    lines.append("")
    text = "\n".join(lines)
    print(text)
    return text

classified_planets_df = classify_planets(catalogues["confirmed"])
print_planets_stats(classified_planets_df)

In [ ]:
# ----------------------------------------------------------------------------
# System-level statistics
# ----------------------------------------------------------------------------

def system_level_stats(df):
    """Group by host star and find systems that contain at least one
    Super-Earth AND at least one Cold Giant."""
    # groups the df by host name and adds new colums with the number of planets in the system, 
    # the number of super-earths, the number of cold giants, and some other properties of the host star
    g = df.groupby("hostname").agg(
        n_planets_in_system=("pl_name", "nunique"),
        n_super_earths=("is_super_earth", "sum"),
        n_cold_giants=("is_cold_giant", "sum"),
        sy_snum=("sy_snum", "first"),
        st_mass=("st_mass", "first"),
        st_met=("st_met", "first"),
        st_spectype=("st_spectype", "first"),
    ).reset_index()

    g["has_super_earth"] = g["n_super_earths"] > 0
    g["has_cold_giant"] = g["n_cold_giants"] > 0
    g["has_both"] = g["has_super_earth"] & g["has_cold_giant"]

    return g

def print_systems_stats(systems):
    lines = []
    lines.append("=" * 70)
    lines.append("SYSTEM-LEVEL STATISTICS")
    lines.append("=" * 70)
    n_systems = len(systems)
    n_multi = (systems["n_planets_in_system"] > 1).sum()
    n_has_se = systems["has_super_earth"].sum()
    n_has_cg = systems["has_cold_giant"].sum()
    n_has_both = systems["has_both"].sum()
    lines.append(f"Total host-star systems: {n_systems:,}")
    lines.append(f"Multi-planet systems:    {n_multi:,}")
    lines.append(f"Systems with >=1 Super-Earth: {n_has_se:,} "
                 f"({100*n_has_se/n_systems:.1f}% of all systems)")
    lines.append(f"Systems with >=1 Cold Giant:  {n_has_cg:,} "
                 f"({100*n_has_cg/n_systems:.1f}% of all systems)")
    lines.append(f"Systems with BOTH Super-Earth(s) AND Cold Giant(s): {n_has_both:,} "
                 f"({100*n_has_both/n_systems:.1f}% of all systems, "
                 f"{100*n_has_both/max(n_multi,1):.1f}% of multi-planet systems)")
    lines.append("")

    both = systems[systems["has_both"]].sort_values(
        "n_planets_in_system", ascending=False)
    if len(both):
        lines.append("Host-star metallicity ([Fe/H]) comparison:")
        for label, mask in [
            ("Systems with Super-Earth + Cold Giant", systems["has_both"]),
            ("All other systems", ~systems["has_both"]),
        ]:
            vals = systems.loc[mask, "st_met"].dropna()
            if len(vals):
                lines.append(f"  {label:<38} n={len(vals):>4}  "
                              f"mean={vals.mean():+.3f}  median={vals.median():+.3f}")
        lines.append("")
        lines.append("Systems with both a Super-Earth and a Cold Giant "
                      f"(showing up to 20 of {len(both)}):")
        for _, row in both.head(20).iterrows():
            lines.append(f"  {row['hostname']:<20} planets={row['n_planets_in_system']:<3} "
                          f"super-earths={int(row['n_super_earths'])} "
                          f"cold-giants={int(row['n_cold_giants'])}")

    text = "\n".join(lines)
    print(text)
    return text

systems_df = system_level_stats(classified_planets_df)
print_systems_stats(systems_df)

In [ ]:
# ----------------------------------------------------------------------------
# Plots
# ----------------------------------------------------------------------------
# planet radius distribution plot

fig, ax = plt.subplots(figsize=(8, 5))
rade = classified_planets_df["pl_rade"].dropna()
bins = np.logspace(np.log10(max(rade.min(), 0.3)), np.log10(rade.max()), 60)
ax.hist(rade, bins=bins, color=COLORS["other"], edgecolor="white", label="All confirmed planets")
ax.hist(classified_planets_df.loc[classified_planets_df["is_super_earth"], "pl_rade"].dropna(), bins=bins,
        color=COLORS["super_earth"], edgecolor="white", label="Super-Earths")
ax.hist(classified_planets_df.loc[classified_planets_df["is_cold_giant"], "pl_rade"].dropna(), bins=bins,
        color=COLORS["cold_giant"], edgecolor="white", label="Cold giants", alpha=0.85)
ax.set_xscale("log")
ax.set_xlabel("$R_{\\rm planet}$ [$R_{\oplus}$]")
ax.set_ylabel("Count")
ax.set_title("Planet radius distribution")
ax.set_xlim(rade.min(), rade.max())
print("Min and max radius of planets: ",rade.min(), rade.max())
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "radius_distribution.png"))

In [ ]:

def plot_mass_radius(df):
    fig, ax = plt.subplots(1,3, figsize=(18, 6))
    other = df[df["pl_category"] == "other"]
    se = df[df["is_super_earth"]]
    cg = df[df["is_cold_giant"]]
    # radius mass plot
    ax[0].scatter(other["pl_rade"], other["pl_bmasse"], s=10, alpha=0.25,
               color=COLORS["other"], label="Other planets")
    ax[0].scatter(se["pl_rade"], se["pl_bmasse"], s=16, alpha=0.8,
               color=COLORS["super_earth"], label="Super-Earths")
    ax[0].scatter(cg["pl_rade"], cg["pl_bmasse"], s=16, alpha=0.8,
               color=COLORS["cold_giant"], label="Cold giants")
    # position mass plot
    ax[1].scatter(other["pl_orbsmax"], other["pl_bmasse"], s=10, alpha=0.25,
               color=COLORS["other"], label="Other planets")
    ax[1].scatter(se["pl_orbsmax"], se["pl_bmasse"], s=16, alpha=0.8,
               color=COLORS["super_earth"], label="Super-Earths")
    ax[1].scatter(cg["pl_orbsmax"], cg["pl_bmasse"], s=16, alpha=0.8,
               color=COLORS["cold_giant"], label="Cold giants")
    # position radius plot
    ax[2].scatter(other["pl_orbsmax"], other["pl_rade"], s=10, alpha=0.25,
               color=COLORS["other"], label="Other planets")
    ax[2].scatter(se["pl_orbsmax"], se["pl_rade"], s=16, alpha=0.8,
               color=COLORS["super_earth"], label="Super-Earths")
    ax[2].scatter(cg["pl_orbsmax"], cg["pl_rade"], s=16, alpha=0.8,
               color=COLORS["cold_giant"], label="Cold giants") 
    
    for a in ax:
        a.set_xscale("log")
        a.set_yscale("log")
        a.legend()
        a.set_ylabel("$M_{\\rm planet}$ [$M_{\oplus}$]")

    ax[0].set_xlabel("$R_{\\rm planet}$ [$R_{\oplus}$]")
    ax[0].set_title("Mass-radius diagram")
    ax[1].set_xlabel("$a_{\\rm planet}$ [AU]")
    ax[1].set_title("Mass-orbital distance diagram")
    ax[2].set_xlabel("$a_{\\rm planet}$ [AU]")
    ax[2].set_ylabel("$R_{\\rm planet}$ [$R_{\oplus}$]")

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "mass_radius_scatter.png"))

plot_mass_radius(classified_planets_df)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

def plot_mass_radius_3d(df):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    other = df[df["pl_category"] == "other"]
    se = df[df["is_super_earth"]]
    cg = df[df["is_cold_giant"]]

    ax.scatter(other["pl_orbsmax"], other["pl_bmasse"], other["pl_rade"],
               s=10, alpha=0.25, color=COLORS["other"], label="Other planets")
    ax.scatter(se["pl_orbsmax"], se["pl_bmasse"], se["pl_rade"],
               s=16, alpha=0.8, color=COLORS["super_earth"], label="Super-Earths")
    ax.scatter(cg["pl_orbsmax"], cg["pl_bmasse"], cg["pl_rade"],
               s=16, alpha=0.8, color=COLORS["cold_giant"], label="Cold giants")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_zscale("log")

    ax.set_xlabel("$a_{\\rm planet}$ [AU]")
    ax.set_ylabel("$M_{\\rm planet}$ [$M_{\oplus}$]")
    ax.set_zlabel("$R_{\\rm planet}$ [$R_{\oplus}$]")
    ax.set_title("Mass-radius-position diagram")
    ax.legend()

    # much more transparent grid
    ax.xaxis._axinfo["grid"]["color"] = (0.5, 0.5, 0.5, 0.1)
    ax.yaxis._axinfo["grid"]["color"] = (0.5, 0.5, 0.5, 0.1)
    ax.zaxis._axinfo["grid"]["color"] = (0.5, 0.5, 0.5, 0.1)

    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "mass_radius_position_3d.png"))
    return fig, ax

plot_mass_radius_3d(classified_planets_df)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def plot_mass_radius_3d_rotating(df, n_frames=120, interval=50):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")

    other = df[df["pl_category"] == "other"]
    se = df[df["is_super_earth"]]
    cg = df[df["is_cold_giant"]]

    ax.scatter(other["pl_orbsmax"], other["pl_bmasse"], other["pl_rade"],
               s=10, alpha=0.25, color=COLORS["other"], label="Other planets")
    ax.scatter(se["pl_orbsmax"], se["pl_bmasse"], se["pl_rade"],
               s=16, alpha=0.8, color=COLORS["super_earth"], label="Super-Earths")
    ax.scatter(cg["pl_orbsmax"], cg["pl_bmasse"], cg["pl_rade"],
               s=16, alpha=0.8, color=COLORS["cold_giant"], label="Cold giants")

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_zscale("log")

    ax.set_xlabel("$a_{\\rm planet}$ [AU]")
    ax.set_ylabel("$M_{\\rm planet}$ [$M_{\oplus}$]")
    ax.set_zlabel("$R_{\\rm planet}$ [$R_{\oplus}$]")
    ax.set_title("Mass-radius-position diagram")
    ax.legend()

    def update(frame):
        ax.view_init(elev=20, azim=frame * (360 / n_frames))
        return fig,

    ani = FuncAnimation(fig, update, frames=n_frames, interval=interval, blit=False)
    plt.close(fig)  # prevents a static duplicate frame from also being displayed
    return ani

ani = plot_mass_radius_3d_rotating(classified_planets_df)
HTML(ani.to_jshtml())


In [ ]:
def plot_both_systems_summary(systems):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))

    categories = ["Super-Earth\nonly", "Cold giant\nonly", "Both", "Neither"]
    se_only = (systems["has_super_earth"] & ~systems["has_cold_giant"]).sum()
    cg_only = (systems["has_cold_giant"] & ~systems["has_super_earth"]).sum()
    both = systems["has_both"].sum()
    neither = (~systems["has_super_earth"] & ~systems["has_cold_giant"]).sum()
    values = [se_only, cg_only, both, neither]
    colors = [COLORS["super_earth"], COLORS["cold_giant"], COLORS["both"], COLORS["other"]]
    axes[0].bar(categories, values, color=colors, edgecolor="white")
    for i, v in enumerate(values):
        axes[0].text(i, v + max(values) * 0.01, f"{v:,}", ha="center", fontsize=9)
    axes[0].set_ylabel("Number of systems")
    axes[0].set_title("Host-star systems by planet content")

    met_both = systems.loc[systems["has_both"], "st_met"].dropna()
    met_other = systems.loc[~systems["has_both"], "st_met"].dropna()
    if len(met_both) and len(met_other):
        axes[1].boxplot([met_both, met_other], tick_labels=["SE + Cold\nGiant systems", "Other\nsystems"])
        axes[1].set_ylabel("Host star [Fe/H]")
        axes[1].set_title("Host star metallicity comparison")
    else:
        axes[1].axis("off")
        axes[1].text(0.5, 0.5, "Not enough metallicity data", ha="center")

    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "super_earth_cold_giant_systems.png"))

plot_both_systems_summary(systems_df)

In [ ]:
def plot_se_period_vs_cg_mass(df, classified_planets_df):
    both_systems = df[df["has_both"]]
    se_periods = []
    cg_masses = []
    hostnames = []

    for hostname in both_systems["hostname"]:
        system_planets = classified_planets_df[classified_planets_df["hostname"] == hostname]
        se_planets = system_planets[system_planets["is_super_earth"]]
        cg_planets = system_planets[system_planets["is_cold_giant"]]

        cg_mass = cg_planets["pl_bmasse"].dropna().max()
        if pd.isna(cg_mass):
            continue

        for period in se_planets["pl_orbper"].dropna():
            se_periods.append(period)
            cg_masses.append(cg_mass)
            hostnames.append(hostname)

    return se_periods, cg_masses, hostnames


se_periods, cg_masses, hostnames = plot_se_period_vs_cg_mass(systems_df, classified_planets_df)

# one color per unique system, drawn from inferno
unique_hosts = sorted(set(hostnames))
cmap = plt.colormaps["inferno"].resampled(len(unique_hosts))
host_to_color = {h: cmap(i) for i, h in enumerate(unique_hosts)}
point_colors = [host_to_color[h] for h in hostnames]

fig, ax = plt.subplots(figsize=(7, 6))

# connect same-system SEs with a line (period now on x-axis)
for h in unique_hosts:
    idx = [i for i, hh in enumerate(hostnames) if hh == h]
    if len(idx) > 1:
        xs = sorted(se_periods[i] for i in idx)
        ys = [cg_masses[i] for i in idx]  # constant per system
        ax.plot(xs, ys, color=host_to_color[h], alpha=0.5, lw=1.5, zorder=1)

# scatter on top of the lines
ax.scatter(se_periods, cg_masses, s=25, alpha=0.9, color=point_colors, zorder=2)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("$P_{\\rm super\\text{-}Earth}$ [days]")
ax.set_ylabel("$M_{\\rm cold\\ giant}$ [$M_{\oplus}$]")
ax.set_title("Super-Earth period vs. its system's cold-giant mass")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "se_period_vs_cg_mass.png"))

## Playing around
My script to understand the catalogue

In [ ]:
df = load_data(CSV_PATH)

print(df.shape)
print(df['discoverymethod'].value_counts())          # compare against method


In [ ]:
microlensing_df = df[df['discoverymethod'] == 'Microlensing']
astrometry_df = df[df['discoverymethod'] == 'Astrometry']
imaging_df = df[df['discoverymethod'] == 'Imaging']
transit_df = df[df['discoverymethod'] == 'Transit']
rv_df = df[df['discoverymethod'] == 'Radial Velocity']

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(microlensing_df['pl_orbsmax'],microlensing_df['pl_bmasse'],  s=10, alpha=0.5, color='green', label='Microlensing')
ax.scatter(astrometry_df['pl_orbsmax'],astrometry_df['pl_bmasse'],  s=10, alpha=0.5, color='purple', label='Astrometry')
ax.scatter(imaging_df['pl_orbsmax'],imaging_df['pl_bmasse'],  s=10, alpha=0.5, color='blue', label='Imaging')
ax.scatter(transit_df['pl_orbsmax'],transit_df['pl_bmasse'],  s=10, alpha=0.5, color='red', label='Transit')
ax.scatter(rv_df['pl_orbsmax'],rv_df['pl_bmasse'],  s=10, alpha=0.5, color='orange', label='Radial Velocity')
ax.set_ylabel("Planet mass [$M_{\oplus}$]", size=12)
ax.set_xlabel("Semi-major axis [AU]", size=12)
ax.tick_params(axis='both', which='major', labelsize=12, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, direction='in')
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend()


## Multiplanet systems
slice of the catalogue with only multiplanet systems

In [ ]:
# Keep only the default/reference row per planet
df_unique = df[df['default_flag'] == 1].copy()
# Filter the original df to just multiplanet systems (more than one planet per host star)
multi_planet_df = df_unique[df_unique.groupby('hostname')['pl_name'].transform('nunique') > 1]
print(f"Number of multiplanet systems: {multi_planet_df['hostname'].nunique()}")
fig, ax = plt.subplots(figsize=(7, 5))
for hostname, group in multi_planet_df.groupby('hostname'):
    ax.plot(group['pl_orbsmax'], group['pl_bmasse'], marker='o', linestyle='-', alpha=0.5, label=hostname)
ax.set_ylabel("$M_{\\rm planet}$ [$M_{\oplus}$]", size=12)
ax.set_xlabel("$a_{\\rm planet}$ [AU]", size=12)
ax.tick_params(axis='both', which='major', labelsize=12, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, direction='in')
ax.set_xscale("log")    
ax.set_yscale("log")


In [ ]:
#color code them by discovery method
print("number of multiplanet systems discovered by microlensing:", multi_planet_df[multi_planet_df['discoverymethod'] == 'Microlensing']['hostname'].nunique())
print("number of multiplanet systems discovered by astrometry:", multi_planet_df[multi_planet_df['discoverymethod'] == 'Astrometry']['hostname'].nunique())
print("number of multiplanet systems discovered by imaging:", multi_planet_df[multi_planet_df['discoverymethod'] == 'Imaging']['hostname'].nunique())
print("number of multiplanet systems discovered by transit:", multi_planet_df[multi_planet_df['discoverymethod'] == 'Transit']['hostname'].nunique())
print("number of multiplanet systems discovered by radial velocity:", multi_planet_df[multi_planet_df['discoverymethod'] == 'Radial Velocity']['hostname'].nunique())


fig, ax = plt.subplots(figsize=(7, 5))
for hostname, group in multi_planet_df.groupby('hostname'):
    discovery_method = group['discoverymethod'].iloc[0]
    if discovery_method == 'Microlensing':
        color = 'green'
    elif discovery_method == 'Astrometry':
        color = 'gold'
    elif discovery_method == 'Imaging':
        color = 'blue'
    elif discovery_method == 'Transit':
        color = 'orange'
    elif discovery_method == 'Radial Velocity':
        color = 'red'
    else:
        color = 'gray'  # default color for other methods
    ax.plot(group['pl_orbsmax'], group['pl_bmasse'], marker='o', linestyle='-', alpha=0.5, label=hostname, color=color) 
ax.set_ylabel("$M_{\\rm planet}$ [$M_{\oplus}$]", size=12)
ax.set_xlabel("$a_{\\rm planet}$ [AU]", size=12)
ax.tick_params(axis='both', which='major', labelsize=12, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, direction='in')
ax.set_xscale("log")
ax.set_yscale("log")

In [ ]:
## cut down to systems that only have super-Earths and cold giants, and plot them
SE_mass_min, SE_mass_max = 2.0, 20.0
CG_mass_min = 95.0
SE_pos_max = 10
CG_pos_min = 1e-2

# Step 1: classify each planet using mass AND position together
is_SE = df_unique['pl_bmasse'].between(SE_mass_min, SE_mass_max) & (df_unique['pl_orbsmax'] <= SE_pos_max)
is_CG = (df_unique['pl_bmasse'] >= CG_mass_min) & (df_unique['pl_orbsmax'] >= CG_pos_min)

df_unique = df_unique.copy()
df_unique['planet_type'] = np.select([is_SE, is_CG], ['SE', 'CG'], default='other')

# Step 2: keep systems where every planet is SE or CG (none classified as 'other'),
# AND require at least one of each type present
def is_SE_CG_only_system(x):
    types = x['planet_type']
    no_other = (types != 'other').all()
    has_both = (types == 'SE').any() and (types == 'CG').any()
    return no_other and has_both

SE_CG_systems = df_unique.groupby('hostname').filter(is_SE_CG_only_system)

fig, ax = plt.subplots(figsize=(7, 5))
for hostname, group in SE_CG_systems.groupby('hostname'):
    discovery_method = group['discoverymethod'].iloc[0]
    if discovery_method == 'Microlensing':
        color = 'green'
    elif discovery_method == 'Astrometry':
        color = 'gold'
    elif discovery_method == 'Imaging':
        color = 'blue'
    elif discovery_method == 'Transit':
        color = 'orange'
    elif discovery_method == 'Radial Velocity':
        color = 'red'
    else:
        color = 'gray'  # default color for other methods
    ax.plot(group['pl_orbsmax'], group['pl_bmasse'], marker='o', linestyle='-', alpha=0.5, label=hostname, color=color)

#overplot in grey all the SE from transit and rv
ax.scatter(df_unique[(df_unique['planet_type'] == 'SE') & (df_unique['discoverymethod'].isin(['Transit', 'Radial Velocity']))]['pl_orbsmax'],
           df_unique[(df_unique['planet_type'] == 'SE') & (df_unique['discoverymethod'].isin(['Transit', 'Radial Velocity']))]['pl_bmasse'],
           s=20, alpha=0.3, color='gray', label='SE from Transit/RV')
#overplot the CG from transit and rv
ax.scatter(df_unique[(df_unique['planet_type'] == 'CG') & (df_unique['discoverymethod'].isin(['Transit', 'Radial Velocity']))]['pl_orbsmax'],
           df_unique[(df_unique['planet_type'] == 'CG') & (df_unique['discoverymethod'].isin(['Transit', 'Radial Velocity']))]['pl_bmasse'],
           s=20, alpha=0.3, color='gray', label='CG from Transit/RV')


ax.set_ylabel("Planet mass [$M_{\oplus}$]", size=12)
ax.set_xlabel("Semi-major axis [AU]", size=12)
ax.tick_params(axis='both', which='major', labelsize=12, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, direction='in')
ax.set_xscale("log")
ax.set_yscale("log")    


In [ ]:
# mass distribution of the SE and CG in general and in the SE+CG systems
fig, ax = plt.subplots(figsize=(7, 5))
bins = np.logspace(np.log10(1), np.log10(1000), 50)
ax.hist(df_unique[df_unique['planet_type'] == 'SE']['pl_bmasse'].dropna(), bins=bins, color='orange', alpha=0.5, label='All SE', edgecolor='white')
ax.hist(df_unique[df_unique['planet_type'] == 'CG']['pl_bmasse'].dropna(), bins=bins, color='blue', alpha=0.5, label='All CG', edgecolor='white')
ax.hist(SE_CG_systems[SE_CG_systems['planet_type'] == 'SE']['pl_bmasse'].dropna(), bins=bins, color='orange', alpha=0.9, label='SE in SE+CG systems', edgecolor='white')
ax.hist(SE_CG_systems[SE_CG_systems['planet_type'] == 'CG']['pl_bmasse'].dropna(), bins=bins, color='blue', alpha=0.9, label='CG in SE+CG systems', edgecolor='white')
ax.set_xscale("log")
ax.set_xlabel("Planet mass [$M_{\oplus}$]", size=12)
ax.set_ylabel("Count", size=12)
ax.tick_params(axis='both', which='major', labelsize=12, direction='in')
ax.tick_params(axis='both', which='minor', labelsize=10, direction='in')
ax.legend() 

#fit to the data to see if the SE and CG in the SE+CG systems are drawn from the same distribution as the general population
from scipy.stats import ks_2samp
SE_all = df_unique[df_unique['planet_type'] == 'SE']['pl_bmasse'].dropna()
SE_in_SE_CG = SE_CG_systems[SE_CG_systems['planet_type'] == 'SE']['pl_bmasse'].dropna()
CG_all = df_unique[df_unique['planet_type'] == 'CG']['pl_bmasse'].dropna()
CG_in_SE_CG = SE_CG_systems[SE_CG_systems['planet_type'] == 'CG']['pl_bmasse'].dropna()
